# DualCrossVitYolo — Expériences KAN
Ce notebook permet de lancer un entraînement pour chaque variante KAN.
**Seule la cellule `Parameters` est à modifier entre deux runs.**

## Install

In [ ]:
!pip install -q git+https://github.com/Victand/pfe_crossvit_dual.git@code_rework --no-deps


## Parameters

| `KAN_MODE`          | Description                                             | Risque overfitting |
|---------------------|---------------------------------------------------------|--------------------|
| `"none"`            | Baseline — architecture originale                       | —                  |
| `"head"`            | Option 1 — KANLinear remplace le head ✅ recommandé     | faible             |
| `"bottleneck"`      | Option 2 — couche KAN résiduelle avant le head          | moyen              |
| `"head+bottleneck"` | Options 1 + 2 combinées                                 | moyen              |
| `"ffn"`             | Option 3 — FFN du dernier MultiScaleBlock remplacé      | élevé              |

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET      = "thorns"   # "genera" ou "thorns"
CLEAR_OUTPUT = False
CLEAR_CACHE  = False

# ── Variante KAN ──────────────────────────────────────────────────────────────
# Modifier uniquement cette section entre deux runs
KAN_MODE            = "none"   # "none" | "head" | "bottleneck" | "head+bottleneck" | "ffn"
KAN_GRID_SIZE       = 5        # expressivité des splines (3–8)
KAN_BOTTLENECK_DIM  = 64       # dim cachée bottleneck (option "bottleneck" uniquement)
KAN_FFN_LAST_ONLY   = True     # True = remplace seulement le dernier bloc (option "ffn")

In [ ]:
parameters = {
    "resume_path": None,
    "lr": 0.0001,
    "lr_factors": {0: 1.0, 1: 0.5, 2: 0.3, 3: 0.1, 4: 0.05},
    "epochs": 50,
    "patience": 10,
    "batch_size": 8,
    "freeze": True,
    "unfreeze_schedule": {0: 0, 1: 5, 2: 7, 3: 15, 4: 15},
    "alphas": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.1],
    "model": {
        "model_name": "dual_crossvit_yolo",
        "crossvit": "crossvit_15_224",
        "drop_rate": 0.1,
        # KAN — injecté automatiquement depuis la cellule ci-dessus
        "kan_mode":           KAN_MODE,
        "kan_grid_size":      KAN_GRID_SIZE,
        "kan_bottleneck_dim": KAN_BOTTLENECK_DIM,
        "kan_ffn_last_only":  KAN_FFN_LAST_ONLY,
    },
    "dataset": {
        "data_dir": f"/kaggle/input/datasets/ewendano/pfe-{DATASET}/{DATASET}/{DATASET}",
        "train_split": 0.8,
        "n_samples": None,
        "branch_small": "yolo_patches",
        "branch_large": "original",
        "branch_large_weight": "ratio",
        "transforms": [
            "random_affine",
            "random_hflip",
            "random_grayscale",
            "color_jitter",
            "random_erase",
        ],
        "ratio_weight_function": "parabole",
        "ratio_patch_size": 16,
        "yolo_patch_count": 8,
        "yolo_patch_quotas": {0: 2, 1: 0, 2: 6, 3: 0, 4: 0, 5: 0, 6: 0},
        "num_workers": 0,
        "precompute": True,
        "use_cache": True,
        "store_cache": True,
    },
}

## Paths

In [ ]:
import pfe_crossvit_dual.constants.paths as paths
import shutil
import os

paths.OUTPUT_DIR = "/kaggle/working/output/"
paths.CACHE_DIR  = "/kaggle/working/cache/"

for d in [paths.OUTPUT_DIR, paths.CACHE_DIR]:
    if os.path.exists(d) and (CLEAR_OUTPUT if "output" in d else CLEAR_CACHE):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

## Vérification de la configuration

In [ ]:
import torch

print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  → {torch.cuda.get_device_name(0)}")
print(f"\nKAN mode      : {KAN_MODE!r}")
print(f"KAN grid size : {KAN_GRID_SIZE}")
if "bottleneck" in KAN_MODE:
    print(f"KAN bottleneck dim : {KAN_BOTTLENECK_DIM}")
if "ffn" in KAN_MODE:
    print(f"KAN FFN last only  : {KAN_FFN_LAST_ONLY}")

try:
    import efficient_kan
    print("\n✓ efficient-kan installé")
except ImportError:
    if KAN_MODE != "none":
        print("\n✗ efficient-kan manquant — relancez la cellule Install")
    else:
        print("\n─ efficient-kan non requis (mode 'none')")

## Run

In [ ]:
from pfe_crossvit_dual.training.training_pipeline import training_pipeline

training_pipeline(parameters)

## Output

In [ ]:
# Liste les runs disponibles
runs = sorted(os.listdir(paths.OUTPUT_DIR))
print("Runs disponibles :")
for r in runs:
    print(f"  {r}")

In [ ]:
# Zippe un run pour téléchargement
RUN_NAME = runs[-1] if runs else "run_1"  # dernier run par défaut

run_path = os.path.join(paths.OUTPUT_DIR, RUN_NAME)
zip_path = shutil.make_archive(run_path, "zip", run_path)
print(f"Zippé : {zip_path}")